# H1 evaluation: pretrained versus fine-tuned RGB

Evaluates the pretrained and fine-tuned RGB models under the leave-one-site-out design. Model-specific confidence and NMS thresholds are selected on validation AOIs and applied unchanged to each held-out site.

## Setup

Installs the fixed Detectron2 and Detectree2 revisions used in the analysis. Restart the Colab runtime after installation.

In [ ]:
!pip -q install \
    "git+https://github.com/facebookresearch/detectron2.git@a2f4a8771ab77e8411c26b27f24f9489a28a2453"

!pip -q install \
    "git+https://github.com/PatBall1/detectree2.git@d9fb07f0dfb493f34def563c1ff896fecd59210d"

!pip -q install rasterio geopandas

## Paths and configuration

Defines the source-data, model, cache and output paths together with the evaluation parameters.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import shutil
import urllib.request

import cv2
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio

from rasterio.mask import mask as rio_mask
from rasterio.transform import xy as raster_xy
from shapely.geometry import Polygon
from shapely.ops import unary_union

from detectron2.engine import DefaultPredictor
from detectree2.models.train import get_tree_dicts, setup_cfg
from detectree2.preprocessing.tiling import tile_data


ROOT = Path("/content/drive/MyDrive/Congo basin/congo")
WORK = Path("/content/h1_evaluation_work")

STACKS = WORK / "stacks"
CROPS = WORK / "crops"
TILES = WORK / "tiles"
KEEP = WORK / "keep"

RUNS = ROOT / "runs_rgb_v2"
OUTPUTS = ROOT / "h1_evaluation"
PREDICTION_CACHE = OUTPUTS / "candidate_predictions"

REBUILD_INPUTS = False
RECOMPUTE_PREDICTIONS = False

if REBUILD_INPUTS and WORK.exists():
    shutil.rmtree(WORK)

for directory in (
    STACKS,
    CROPS,
    TILES,
    KEEP,
    OUTPUTS,
    PREDICTION_CACHE,
):
    directory.mkdir(parents=True, exist_ok=True)


SITES = ["lokoue", "dzanga", "mbeli"]
STRATA = ["tall", "mid", "small"]

SITE_LABELS = {
    "lokoue": "Lokoué",
    "dzanga": "Dzanga",
    "mbeli": "Mbeli",
}

STRATUM_LABELS = {
    "tall": "High",
    "mid": "Intermediate",
    "small": "Low",
}

TILE_WIDTH = 50
BUFFER = 25

CANDIDATE_THRESHOLD = 0.05
MATCHING_IOU = 0.50

CONFIDENCE_THRESHOLDS = [
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
]

NMS_THRESHOLDS = [
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
]

BASE_MODEL = (
    "COCO-InstanceSegmentation/"
    "mask_rcnn_R_101_FPN_3x.yaml"
)

PRETRAINED_MODEL = Path(
    "/content/230103_randresize_full.pth"
)

paths = {
    site: {
        "rgb": ROOT / site / f"{site}_RGB.TIF",
        "ned": ROOT / site / f"{site}_NED.TIF",
        "stack": STACKS / f"{site}_h1_uint8.tif",
        "mask": ROOT / site / f"mask_{site}.gpkg",
        "aoi": {
            stratum: ROOT / site / f"{site}_aoi_{stratum}.gpkg"
            for stratum in STRATA
        },
        "crowns": {
            stratum: ROOT / site / f"crowns_{site}_{stratum}.gpkg"
            for stratum in STRATA
        },
    }
    for site in SITES
}

## Input validation

Checks the source rasters, spatial annotations and trained-model checkpoints before evaluation.

In [ ]:
required_files = []

for site in SITES:
    required_files.extend([
        paths[site]["rgb"],
        paths[site]["ned"],
        paths[site]["mask"],
        *paths[site]["aoi"].values(),
        *paths[site]["crowns"].values(),
        RUNS / f"rgb_holdout_{site}" / "model_best.pth",
    ])

missing_files = [
    path
    for path in required_files
    if not path.exists()
]

assert not missing_files, (
    "Missing required files:\n"
    + "\n".join(map(str, missing_files))
)

for site in SITES:
    checkpoint = (
        RUNS
        / f"rgb_holdout_{site}"
        / "model_best.pth"
    )

    assert checkpoint.stat().st_size / 1e6 > 400, (
        f"{site}: incomplete fine-tuned checkpoint"
    )

    with (
        rasterio.open(paths[site]["rgb"]) as rgb,
        rasterio.open(paths[site]["ned"]) as ned,
    ):
        assert rgb.count >= 3 and ned.count >= 3
        assert rgb.crs == ned.crs, f"{site}: CRS mismatch"
        assert rgb.shape == ned.shape, f"{site}: shape mismatch"
        assert rgb.transform == ned.transform, (
            f"{site}: transform mismatch"
        )
        assert rgb.bounds == ned.bounds, (
            f"{site}: bounds mismatch"
        )
        assert rgb.res == ned.res, (
            f"{site}: resolution mismatch"
        )
        assert set(rgb.dtypes[:3]) == {"uint16"}
        assert rgb.dtypes[:3] == ned.dtypes[:3]

        print(
            f"{site}: {rgb.width} × {rgb.height} pixels, "
            f"{rgb.res[0]:.3f} m resolution"
        )

if not PRETRAINED_MODEL.exists():
    urllib.request.urlretrieve(
        "https://zenodo.org/records/10522461/"
        "files/230103_randresize_full.pth",
        PRETRAINED_MODEL,
    )

assert PRETRAINED_MODEL.stat().st_size / 1e6 > 400, (
    "Pretrained checkpoint is incomplete"
)

## H1 image preprocessing

Recreates the 8-bit H1 inputs used during training using site- and band-specific 2nd–98th percentile scaling. The RGB pathway reads the first three bands.

In [ ]:
BAND_NAMES = [
    "red",
    "green",
    "blue",
    "near-infrared",
    "red-edge",
    "deep-blue",
]


def build_h1_stack(site):
    output_path = paths[site]["stack"]

    if output_path.exists() and not REBUILD_INPUTS:
        with rasterio.open(output_path) as raster:
            valid_output = (
                raster.count == 6
                and raster.dtypes == ("uint8",) * 6
                and raster.nodata == 0
            )

        if valid_output:
            print(f"{site}: retaining existing H1 stack")
            return output_path

        output_path.unlink()

    with (
        rasterio.open(paths[site]["rgb"]) as rgb,
        rasterio.open(paths[site]["ned"]) as ned,
    ):
        histograms = np.zeros(
            (6, 65536),
            dtype=np.int64,
        )

        for _, window in rgb.block_windows(1):
            values = np.concatenate([
                rgb.read([1, 2, 3], window=window),
                ned.read([1, 2, 3], window=window),
            ], axis=0)

            valid = values.sum(axis=0) > 0

            for band in range(6):
                histograms[band] += np.bincount(
                    values[band][valid]
                    .ravel()
                    .astype(np.int64),
                    minlength=65536,
                )

        lower = np.empty(6, dtype=np.float32)
        upper = np.empty(6, dtype=np.float32)

        for band in range(6):
            # Exclude the nodata bin.
            histograms[band, 0] = 0
            cumulative = np.cumsum(histograms[band])

            assert cumulative[-1] > 0, (
                f"{site}/{BAND_NAMES[band]}: "
                "no valid pixels"
            )

            lower[band] = np.searchsorted(
                cumulative,
                0.02 * cumulative[-1],
            )
            upper[band] = np.searchsorted(
                cumulative,
                0.98 * cumulative[-1],
            )

            assert upper[band] > lower[band], (
                f"{site}/{BAND_NAMES[band]}: "
                "invalid stretch limits"
            )

        metadata = rgb.meta.copy()
        metadata.update(
            driver="GTiff",
            count=6,
            dtype="uint8",
            nodata=0,
            tiled=True,
            blockxsize=512,
            blockysize=512,
            compress="deflate",
        )

        lower_3d = lower.reshape(-1, 1, 1)
        upper_3d = upper.reshape(-1, 1, 1)

        with rasterio.open(
            output_path,
            "w",
            **metadata,
        ) as destination:
            for _, window in rgb.block_windows(1):
                values = np.concatenate([
                    rgb.read([1, 2, 3], window=window),
                    ned.read([1, 2, 3], window=window),
                ], axis=0).astype(np.float32)

                nodata = values.sum(axis=0) == 0

                image = (
                    (values - lower_3d)
                    / (upper_3d - lower_3d)
                    * 253
                    + 1
                )
                image = np.clip(image, 1, 254)
                image[
                    np.broadcast_to(nodata, image.shape)
                ] = 0

                destination.write(
                    image.astype(np.uint8),
                    window=window,
                )

    with rasterio.open(output_path) as raster:
        assert raster.count == 6
        assert raster.dtypes == ("uint8",) * 6
        assert raster.nodata == 0

    print(f"{site}: H1 stack created")
    return output_path


for site in SITES:
    build_h1_stack(site)

## Valid evaluation regions

Reconstructs the masked evaluation regions and summarises the reference crown dataset.

In [ ]:
prepped = {}
dataset_rows = []

for site in SITES:
    with rasterio.open(paths[site]["stack"]) as raster:
        raster_crs = raster.crs

    boundary_mask = gpd.read_file(
        paths[site]["mask"]
    ).to_crs(raster_crs)
    boundary_mask.geometry = (
        boundary_mask.geometry.buffer(0)
    )

    prepped[site] = {}

    for stratum in STRATA:
        aoi = gpd.read_file(
            paths[site]["aoi"][stratum]
        ).to_crs(raster_crs)

        crowns = gpd.read_file(
            paths[site]["crowns"][stratum]
        ).to_crs(raster_crs)

        aoi.geometry = aoi.geometry.buffer(0)
        crowns.geometry = crowns.geometry.buffer(0)

        assert aoi.geometry.is_valid.all()
        assert crowns.geometry.is_valid.all()

        aoi_union = unary_union(aoi.geometry)

        inside_aoi = (
        crowns.geometry.centroid.within(aoi_union)
        )

        assert inside_aoi.all(), (
            f"{site}/{stratum}: crown centroid "
            "outside its AOI"
        )

        keep = gpd.overlay(
            aoi,
            boundary_mask,
            how="difference",
        )
        keep.geometry = keep.geometry.buffer(0)

        assert len(keep) > 0
        assert keep.geometry.is_valid.all()

        keep_path = (
            KEEP
            / f"keep_{site}_{stratum}.gpkg"
        )
        keep.to_file(
            keep_path,
            driver="GPKG",
        )

        keep_union = unary_union(keep.geometry)

        inside_keep_region = (
          crowns.geometry.centroid.within(
            keep_union
          )
        )

        evaluation_crowns = crowns[
          inside_keep_region
        ].copy()

        valid_area_ha = keep_union.area / 10_000
        crown_union = unary_union(
            evaluation_crowns.geometry
        )
        coverage = (
            crown_union.area
            / keep_union.area
            * 100
        )
        density = (
            len(evaluation_crowns)
            / valid_area_ha
        )

        dataset_rows.append({
            "site": SITE_LABELS[site],
            "stratum": STRATUM_LABELS[stratum],
            "valid_area_ha": valid_area_ha,
            "crowns": len(evaluation_crowns),
            "density_ha": density,
            "coverage_pct": coverage,
            "median_crown_area_m2": (
                evaluation_crowns.area.median()
            ),
        })

        prepped[site][stratum] = {
            "crowns": crowns,
            "evaluation_crowns": evaluation_crowns,
            "keep": keep,
            "keep_path": keep_path,
        }

dataset_table = pd.DataFrame(dataset_rows)

assert dataset_table["crowns"].sum() == 1027
assert round(
    dataset_table["valid_area_ha"].sum(),
    1,
) == 78.2

dataset_table.round(3)

## Image tiling

Recreates the overlapping 50 m core and 25 m buffer tiling used for H1 training and evaluation.

In [ ]:
def tile_aoi(site, stratum):
    crowns = prepped[site][stratum]["crowns"]
    keep_path = prepped[site][stratum]["keep_path"]

    aoi = gpd.read_file(
        paths[site]["aoi"][stratum]
    ).to_crs(crowns.crs)

    crop_path = (
        CROPS
        / f"{site}_{stratum}_h1.tif"
    )

    if not crop_path.exists():
        with rasterio.open(
            paths[site]["stack"]
        ) as source:
            image, transform = rio_mask(
                source,
                aoi.geometry.buffer(
                    2,
                    join_style=2,
                ),
                crop=True,
                nodata=0,
            )

            metadata = source.meta.copy()
            metadata.update(
                height=image.shape[1],
                width=image.shape[2],
                transform=transform,
                nodata=0,
            )

        with rasterio.open(
            crop_path,
            "w",
            **metadata,
        ) as destination:
            destination.write(image)

    output_directory = (
        TILES
        / f"{site}_{stratum}_rgb_"
          f"{TILE_WIDTH}_{BUFFER}"
    )

    if not output_directory.exists():
        tile_data(
            img_path=str(crop_path),
            out_dir=str(output_directory),
            buffer=BUFFER,
            tile_width=TILE_WIDTH,
            tile_height=TILE_WIDTH,
            crowns=crowns,
            threshold=0.0,
            nan_threshold=1.0,
            full_coverage=False,
            mode="rgb",
            mask_path=str(keep_path),
            use_convex_mask=False,
            enhance_rgb_contrast=False,
            tile_placement="grid",
            multithreaded=True,
            ignore_bands_indices=[],
        )

    tile_count = len(
        list(output_directory.glob("*.geojson"))
    )

    assert tile_count == 25, (
        f"{site}/{stratum}: expected 25 tiles, "
        f"found {tile_count}"
    )

    print(f"{site}/{stratum}: {tile_count} tiles")
    return output_directory


rgb_directories = {
    (site, stratum): tile_aoi(site, stratum)
    for site in SITES
    for stratum in STRATA
}

total_tiles = sum(
    len(list(directory.glob("*.geojson")))
    for directory in rgb_directories.values()
)

assert total_tiles == 225
print("Total tiles:", total_tiles)

## Geographic reconstruction and matching

Converts tile-level masks to mapped polygons, removes duplicate predictions and performs one-to-one matching at IoU ≥ 0.5.

In [ ]:
def intersection_over_union(first, second):
    if not first.intersects(second):
        return 0.0

    intersection = first.intersection(second).area
    union = (
        first.area
        + second.area
        - intersection
    )

    return intersection / union if union else 0.0


def non_maximum_suppression(predictions, threshold):
    retained = []

    ordered = sorted(
        predictions,
        key=lambda prediction: prediction["score"],
        reverse=True,
    )

    for prediction in ordered:
        duplicate = any(
            intersection_over_union(
                prediction["geometry"],
                retained_prediction["geometry"],
            ) >= threshold
            for retained_prediction in retained
        )

        if not duplicate:
            retained.append(prediction)

    return retained


def greedy_matches(
    predictions,
    references,
    threshold=MATCHING_IOU,
):
    eligible_pairs = []

    for prediction_index, prediction in enumerate(predictions):
        for reference_index, reference in enumerate(references):
            overlap = intersection_over_union(
                prediction,
                reference,
            )

            if overlap >= threshold:
                eligible_pairs.append((
                    overlap,
                    prediction_index,
                    reference_index,
                ))

    eligible_pairs.sort(reverse=True)

    used_predictions = set()
    used_references = set()
    matches = []

    for overlap, prediction_index, reference_index in eligible_pairs:
        if (
            prediction_index in used_predictions
            or reference_index in used_references
        ):
            continue

        used_predictions.add(prediction_index)
        used_references.add(reference_index)

        matches.append((
            overlap,
            prediction_index,
            reference_index,
        ))

    return matches


def precision_recall_f1(tp, fp, fn):
    precision = (
        tp / (tp + fp)
        if tp + fp
        else 0.0
    )
    recall = (
        tp / (tp + fn)
        if tp + fn
        else 0.0
    )
    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall
        else 0.0
    )

    return precision, recall, f1


def mask_to_map_polygons(binary_mask, transform):
    contours, _ = cv2.findContours(
        binary_mask.astype(np.uint8),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE,
    )

    polygons = []

    for contour in contours:
        if len(contour) < 3:
            continue

        coordinates = contour.reshape(-1, 2)

        x, y = raster_xy(
            transform,
            coordinates[:, 1],
            coordinates[:, 0],
            offset="center",
        )

        polygon = Polygon(
            zip(
                np.atleast_1d(x),
                np.atleast_1d(y),
            )
        ).buffer(0)

        if not polygon.is_empty and polygon.area > 0:
            polygons.append(polygon)

    return polygons


def tile_raster_path(record):
    image_path = Path(record["file_name"])

    candidates = [
        image_path.with_suffix(".tif"),
        image_path.with_suffix(".TIF"),
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    matches = list(
        image_path.parent.glob(
            image_path.stem + "*.tif"
        )
    )

    assert len(matches) == 1, (
        f"Could not identify raster for {image_path}"
    )

    return matches[0]

## Reference crowns and evaluation regions

Loads each reference crown once from the source annotations and restricts predictions and references to the same valid regions.

In [ ]:
def reference_records_for_aois(aois):
    records = []

    for site, stratum in aois:
        crowns = prepped[site][stratum][
            "evaluation_crowns"
        ]

        for source_index, geometry in zip(
            crowns.index,
            crowns.geometry,
        ):
            records.append({
                "site": site,
                "stratum": stratum,
                "source_index": int(source_index),
                "geometry": geometry,
            })

    return records


def keep_region_for_aois(aois):
    polygons = []

    for site, stratum in aois:
        keep = prepped[site][stratum]["keep"]

        polygons.extend(
            geometry.buffer(0)
            for geometry in keep.geometry
        )

    return unary_union(polygons)


def evaluate_at(
    predictions,
    reference_records,
    keep_region,
    confidence_threshold,
    nms_threshold,
    return_details=False,
):
    confidence_filtered = [
        prediction
        for prediction in predictions
        if prediction["score"]
        >= confidence_threshold
    ]

    deduplicated = non_maximum_suppression(
        confidence_filtered,
        nms_threshold,
    )

    retained = [
        prediction
        for prediction in deduplicated
        if keep_region.contains(
            prediction["geometry"].centroid
        )
    ]

    references = [
        record["geometry"]
        for record in reference_records
    ]

    matches = greedy_matches(
        [
            prediction["geometry"]
            for prediction in retained
        ],
        references,
    )

    true_positives = len(matches)
    false_positives = (
        len(retained) - true_positives
    )
    false_negatives = (
        len(references) - true_positives
    )

    precision, recall, f1 = precision_recall_f1(
        true_positives,
        false_positives,
        false_negatives,
    )

    metrics = {
        "confidence": confidence_threshold,
        "nms": nms_threshold,
        "tp": true_positives,
        "fp": false_positives,
        "fn": false_negatives,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

    if return_details:
        return metrics, retained, matches

    return metrics

## Candidate prediction generation

Runs inference at a confidence threshold of 0.05 and caches the mapped candidate predictions for subsequent threshold evaluation.

In [ ]:
def build_predictor(weights, holdout):
    train_name, validation_name, _ = (
        register_evaluation_fold(holdout)
    )

    configuration = setup_cfg(
        base_model=BASE_MODEL,
        trains=(train_name,),
        tests=(validation_name,),
        update_model=str(weights),
        workers=2,
        ims_per_batch=2,
        base_lr=0.0003389,
        backbone_freeze=0,
        max_iter=1,
        eval_period=1,
        resize="rand_fixed",
        imgmode="rgb",
        num_bands=3,
        out_dir="/content/h1_evaluation_model",
    )

    configuration.MODEL.WEIGHTS = str(weights)
    configuration.MODEL.ROI_HEADS.SCORE_THRESH_TEST = (
        CANDIDATE_THRESHOLD
    )

    return DefaultPredictor(configuration)


def prediction_cache_path(model_key, site, stratum):
    return (
        PREDICTION_CACHE
        / f"{model_key}_{site}_{stratum}.gpkg"
    )


def read_prediction_cache(model_key, site, stratum):
    cache_path = prediction_cache_path(
        model_key,
        site,
        stratum,
    )

    predictions = gpd.read_file(cache_path)

    return [
        {
            "geometry": row.geometry,
            "score": float(row.score),
            "site": row.site,
            "stratum": row.stratum,
            "tile": row.tile,
        }
        for row in predictions.itertuples()
    ]


def predict_aoi(
    predictor,
    model_key,
    site,
    stratum,
):
    cache_path = prediction_cache_path(
        model_key,
        site,
        stratum,
    )

    if (
        cache_path.exists()
        and not RECOMPUTE_PREDICTIONS
    ):
        print(
            f"{model_key}/{site}/{stratum}: "
            "using cached predictions"
        )

        return read_prediction_cache(
            model_key,
            site,
            stratum,
        )

    predictions = []
    output_crs = None

    tile_records = get_tree_dicts(
        str(rgb_directories[(site, stratum)])
    )

    for record in tile_records:
        raster_path = tile_raster_path(record)

        with rasterio.open(raster_path) as raster:
            transform = raster.transform
            output_crs = raster.crs

        image = cv2.imread(
            record["file_name"],
            cv2.IMREAD_COLOR,
        )

        assert image is not None, (
            f"Could not read {record['file_name']}"
        )

        instances = predictor(image)[
            "instances"
        ].to("cpu")

        masks = instances.pred_masks.numpy()
        scores = instances.scores.numpy()

        for mask, score in zip(masks, scores):
            polygons = mask_to_map_polygons(
                mask.astype(bool),
                transform,
            )

            for polygon in polygons:
                predictions.append({
                    "geometry": polygon,
                    "score": float(score),
                    "site": site,
                    "stratum": stratum,
                    "tile": raster_path.stem,
                })

    assert predictions, (
        f"{model_key}/{site}/{stratum}: "
        "no candidate predictions"
    )

    prediction_table = gpd.GeoDataFrame(
        {
            "score": [
                prediction["score"]
                for prediction in predictions
            ],
            "site": site,
            "stratum": stratum,
            "tile": [
                prediction["tile"]
                for prediction in predictions
            ],
        },
        geometry=[
            prediction["geometry"]
            for prediction in predictions
        ],
        crs=output_crs,
    )

    prediction_table.to_file(
        cache_path,
        driver="GPKG",
    )

    print(
        f"{model_key}/{site}/{stratum}: "
        f"{len(predictions)} candidates saved"
    )

    return predictions

## Fold evaluation

Selects confidence and NMS thresholds by validation F1 and applies them unchanged to the corresponding held-out site and canopy strata.

In [ ]:
from detectron2.data import (
    DatasetCatalog,
    MetadataCatalog,
)


def register_evaluation_fold(holdout):
    tag = f"h1_evaluation_{holdout}"

    dataset_names = {
        split: f"{tag}_{split}"
        for split in ("train", "validation", "test")
    }

    for name in dataset_names.values():
        if name in DatasetCatalog.list():
            DatasetCatalog.remove(name)

        if name in MetadataCatalog.list():
            MetadataCatalog.remove(name)

    partitions = {
        "train": [],
        "validation": [],
        "test": [],
    }

    for (site, stratum), directory in rgb_directories.items():
        records = get_tree_dicts(str(directory))

        if site == holdout:
            partitions["test"].extend(records)
        elif stratum == "mid":
            partitions["validation"].extend(records)
        else:
            partitions["train"].extend(records)

    counts = {
        split: len(records)
        for split, records in partitions.items()
    }

    assert counts == {
        "train": 100,
        "validation": 50,
        "test": 75,
    }, (
        f"{holdout}: unexpected fold composition "
        f"{counts}"
    )

    for split, records in partitions.items():
        name = dataset_names[split]

        DatasetCatalog.register(
            name,
            lambda records=records: records,
        )

        MetadataCatalog.get(name).set(
            thing_classes=["tree"]
        )

    return (
        dataset_names["train"],
        dataset_names["validation"],
        dataset_names["test"],
    )


def fold_aois(holdout):
    development_sites = [
        site
        for site in SITES
        if site != holdout
    ]

    validation_aois = [
        (site, "mid")
        for site in development_sites
    ]

    test_aois = [
        (holdout, stratum)
        for stratum in STRATA
    ]

    assert not (
        set(validation_aois)
        & set(test_aois)
    )

    return validation_aois, test_aois


def collect_predictions(
    predictor,
    model_key,
    aois,
):
    return [
        prediction
        for site, stratum in aois
        for prediction in predict_aoi(
            predictor,
            model_key,
            site,
            stratum,
        )
    ]


def select_thresholds(
    predictions,
    validation_aois,
):
    references = reference_records_for_aois(
        validation_aois
    )
    keep_region = keep_region_for_aois(
        validation_aois
    )

    grid_results = [
        evaluate_at(
            predictions,
            references,
            keep_region,
            confidence,
            nms,
        )
        for confidence in CONFIDENCE_THRESHOLDS
        for nms in NMS_THRESHOLDS
    ]

    best = max(
        grid_results,
        key=lambda result: (
            result["f1"],
            result["confidence"],
            result["nms"],
        ),
    )

    return best, grid_results


def run_fold(
    model_name,
    model_key,
    weights,
    holdout,
):
    validation_aois, test_aois = fold_aois(
        holdout
    )

    predictor = build_predictor(
        weights,
        holdout,
    )

    validation_predictions = collect_predictions(
        predictor,
        model_key,
        validation_aois,
    )

    test_predictions = collect_predictions(
        predictor,
        model_key,
        test_aois,
    )

    selected, validation_grid = select_thresholds(
        validation_predictions,
        validation_aois,
    )

    test_metrics = evaluate_at(
        test_predictions,
        reference_records_for_aois(test_aois),
        keep_region_for_aois(test_aois),
        selected["confidence"],
        selected["nms"],
    )

    stratum_metrics = {}

    for stratum in STRATA:
        stratum_aoi = [(holdout, stratum)]

        stratum_predictions = [
            prediction
            for prediction in test_predictions
            if prediction["stratum"] == stratum
        ]

        stratum_metrics[stratum] = evaluate_at(
            stratum_predictions,
            reference_records_for_aois(
                stratum_aoi
            ),
            keep_region_for_aois(
                stratum_aoi
            ),
            selected["confidence"],
            selected["nms"],
        )

    print(
        f"{model_name}/{holdout}: "
        f"confidence={selected['confidence']:.2f}, "
        f"NMS={selected['nms']:.2f}, "
        f"P={test_metrics['precision']:.3f}, "
        f"R={test_metrics['recall']:.3f}, "
        f"F1={test_metrics['f1']:.3f}"
    )

    return {
        "model": model_name,
        "holdout": holdout,
        "selection": selected,
        "validation_grid": validation_grid,
        "test": test_metrics,
        "strata": stratum_metrics,
    }

## Run the H1 evaluation

Evaluates the pretrained checkpoint and the corresponding fine-tuned RGB model in each leave-one-site-out fold.

In [ ]:
fine_tuned_results = {}
pretrained_results = {}

for holdout in SITES:
    print(f"\nHold out: {SITE_LABELS[holdout]}")

    fine_tuned_results[holdout] = run_fold(
        model_name="Fine-tuned RGB",
        model_key=f"finetuned_{holdout}",
        weights=(
            RUNS
            / f"rgb_holdout_{holdout}"
            / "model_best.pth"
        ),
        holdout=holdout,
    )

    pretrained_results[holdout] = run_fold(
        model_name="Pretrained RGB",
        model_key="pretrained",
        weights=PRETRAINED_MODEL,
        holdout=holdout,
    )

## Results and verification

Calculates site-level and macro-averaged performance and verifies the results against the reported H1 values.

In [ ]:
def macro_average(results):
    return {
        metric: float(np.mean([
            results[site]["test"][metric]
            for site in SITES
        ]))
        for metric in (
            "precision",
            "recall",
            "f1",
        )
    }


pretrained_macro = macro_average(
    pretrained_results
)
fine_tuned_macro = macro_average(
    fine_tuned_results
)

expected_site_results = {
    "lokoue": {
        "pretrained": (0.328, 0.171, 0.225),
        "fine_tuned": (0.620, 0.298, 0.402),
    },
    "dzanga": {
        "pretrained": (0.286, 0.183, 0.223),
        "fine_tuned": (0.426, 0.463, 0.444),
    },
    "mbeli": {
        "pretrained": (0.217, 0.336, 0.264),
        "fine_tuned": (0.466, 0.498, 0.481),
    },
}

for site, expected in expected_site_results.items():
    for result_name, result_set in (
        ("pretrained", pretrained_results),
        ("fine_tuned", fine_tuned_results),
    ):
        observed = result_set[site]["test"]
        expected_values = expected[result_name]

        for metric, expected_value in zip(
            ("precision", "recall", "f1"),
            expected_values,
        ):
            assert abs(
                observed[metric] - expected_value
            ) < 0.0015, (
                f"{result_name}/{site}/{metric}: "
                f"{observed[metric]:.6f} != "
                f"{expected_value:.3f}"
            )

expected_macro = {
    "pretrained": (0.277, 0.230, 0.237),
    "fine_tuned": (0.504, 0.419, 0.442),
}

for result_name, observed in (
    ("pretrained", pretrained_macro),
    ("fine_tuned", fine_tuned_macro),
):
    for metric, expected_value in zip(
        ("precision", "recall", "f1"),
        expected_macro[result_name],
    ):
        assert abs(
            observed[metric] - expected_value
        ) < 0.0015

expected_stratum_f1 = {
    "lokoue": {
        "tall": 0.354,
        "mid": 0.433,
        "small": 0.407,
    },
    "dzanga": {
        "tall": 0.431,
        "mid": 0.412,
        "small": 0.498,
    },
    "mbeli": {
        "tall": 0.452,
        "mid": 0.636,
        "small": 0.208,
    },
}

for site in SITES:
    for stratum in STRATA:
        observed = fine_tuned_results[
            site
        ]["strata"][stratum]["f1"]

        expected = expected_stratum_f1[
            site
        ][stratum]

        assert abs(observed - expected) < 0.0015

print("PASS: H1 results match the dissertation.")

## Export results

Exports the dataset summary, H1 performance results and validation-selected thresholds for the combined analysis.

In [ ]:
site_rows = []

for site in SITES:
    pretrained = pretrained_results[site]["test"]
    fine_tuned = fine_tuned_results[site]["test"]

    # Match the displayed three-decimal ΔF1 values.
    delta_f1 = (
        round(fine_tuned["f1"], 3)
        - round(pretrained["f1"], 3)
    )

    site_rows.append({
        "site": SITE_LABELS[site],
        "pretrained_precision": pretrained["precision"],
        "pretrained_recall": pretrained["recall"],
        "pretrained_f1": pretrained["f1"],
        "fine_tuned_precision": fine_tuned["precision"],
        "fine_tuned_recall": fine_tuned["recall"],
        "fine_tuned_f1": fine_tuned["f1"],
        "delta_f1": delta_f1,
    })

site_rows.append({
    "site": "Macro",
    "pretrained_precision": pretrained_macro["precision"],
    "pretrained_recall": pretrained_macro["recall"],
    "pretrained_f1": pretrained_macro["f1"],
    "fine_tuned_precision": fine_tuned_macro["precision"],
    "fine_tuned_recall": fine_tuned_macro["recall"],
    "fine_tuned_f1": fine_tuned_macro["f1"],
    "delta_f1": (
        round(fine_tuned_macro["f1"], 3)
        - round(pretrained_macro["f1"], 3)
    ),
})

site_results_table = pd.DataFrame(site_rows)


stratum_rows = []

for site in SITES:
    for stratum in STRATA:
        stratum_rows.append({
            "site": SITE_LABELS[site],
            "stratum": STRATUM_LABELS[stratum],
            "pretrained_f1": (
                pretrained_results[site]
                ["strata"][stratum]["f1"]
            ),
            "fine_tuned_f1": (
                fine_tuned_results[site]
                ["strata"][stratum]["f1"]
            ),
        })

stratum_results_table = pd.DataFrame(
    stratum_rows
)


threshold_rows = []

for model_name, results in (
    ("Pretrained RGB", pretrained_results),
    ("Fine-tuned RGB", fine_tuned_results),
):
    for site in SITES:
        selection = results[site]["selection"]

        threshold_rows.append({
            "model": model_name,
            "held_out_site": SITE_LABELS[site],
            "confidence": selection["confidence"],
            "nms_iou": selection["nms"],
            "validation_f1": selection["f1"],
        })

threshold_table = pd.DataFrame(threshold_rows)


dataset_table.to_csv(
    OUTPUTS / "dataset_composition.csv",
    index=False,
)

site_results_table.to_csv(
    OUTPUTS / "h1_site_results.csv",
    index=False,
)

stratum_results_table.to_csv(
    OUTPUTS / "h1_stratum_results.csv",
    index=False,
)

threshold_table.to_csv(
    OUTPUTS / "h1_threshold_selection.csv",
    index=False,
)

with open(
    OUTPUTS / "results_h1_rgb.json",
    "w",
) as output_file:
    json.dump(
        {
            "protocol": {
                "candidate_confidence": (
                    CANDIDATE_THRESHOLD
                ),
                "matching_iou": MATCHING_IOU,
                "confidence_grid": (
                    CONFIDENCE_THRESHOLDS
                ),
                "nms_grid": NMS_THRESHOLDS,
                "threshold_selection": (
                    "per-model, per-fold validation "
                    "F1; frozen before testing"
                ),
            },
            "pretrained": pretrained_results,
            "fine_tuned": fine_tuned_results,
            "macro": {
                "pretrained": pretrained_macro,
                "fine_tuned": fine_tuned_macro,
            },
        },
        output_file,
        indent=2,
    )

display(site_results_table.round(3))
display(stratum_results_table.round(3))
display(threshold_table.round(3))

print("Outputs written to:", OUTPUTS)

## Figure 3 evaluation layers

Classifies held-out Dzanga predictions and references using the frozen fold thresholds and exports the mapped layers used to prepare Figure 3.

In [ ]:
dzanga_aois = [
    ("dzanga", stratum)
    for stratum in STRATA
]

dzanga_candidates = [
    prediction
    for stratum in STRATA
    for prediction in read_prediction_cache(
        "finetuned_dzanga",
        "dzanga",
        stratum,
    )
]

dzanga_references = reference_records_for_aois(
    dzanga_aois
)

dzanga_selection = fine_tuned_results[
    "dzanga"
]["selection"]

(
    dzanga_metrics,
    dzanga_predictions,
    dzanga_matches,
) = evaluate_at(
    dzanga_candidates,
    dzanga_references,
    keep_region_for_aois(dzanga_aois),
    dzanga_selection["confidence"],
    dzanga_selection["nms"],
    return_details=True,
)

assert dzanga_metrics["tp"] == 210
assert dzanga_metrics["fp"] == 283
assert dzanga_metrics["fn"] == 244

matched_predictions = {
    prediction_index: overlap
    for (
        overlap,
        prediction_index,
        reference_index,
    ) in dzanga_matches
}

matched_references = {
    reference_index: overlap
    for (
        overlap,
        prediction_index,
        reference_index,
    ) in dzanga_matches
}

with rasterio.open(
    paths["dzanga"]["stack"]
) as raster:
    dzanga_crs = raster.crs


prediction_layer = gpd.GeoDataFrame(
    {
        "status": [
            (
                "true_positive"
                if index in matched_predictions
                else "false_positive"
            )
            for index in range(
                len(dzanga_predictions)
            )
        ],
        "score": [
            prediction["score"]
            for prediction in dzanga_predictions
        ],
        "match_iou": [
            matched_predictions.get(
                index,
                np.nan,
            )
            for index in range(
                len(dzanga_predictions)
            )
        ],
        "stratum": [
            STRATUM_LABELS[
                prediction["stratum"]
            ]
            for prediction in dzanga_predictions
        ],
    },
    geometry=[
        prediction["geometry"]
        for prediction in dzanga_predictions
    ],
    crs=dzanga_crs,
)


reference_layer = gpd.GeoDataFrame(
    {
        "status": [
            (
                "matched"
                if index in matched_references
                else "false_negative"
            )
            for index in range(
                len(dzanga_references)
            )
        ],
        "match_iou": [
            matched_references.get(
                index,
                np.nan,
            )
            for index in range(
                len(dzanga_references)
            )
        ],
        "stratum": [
            STRATUM_LABELS[
                reference["stratum"]
            ]
            for reference in dzanga_references
        ],
        "source_index": [
            reference["source_index"]
            for reference in dzanga_references
        ],
    },
    geometry=[
        reference["geometry"]
        for reference in dzanga_references
    ],
    crs=dzanga_crs,
)


figure_layers = (
    OUTPUTS
    / "H1_dzanga_figure_layers.gpkg"
)

if figure_layers.exists():
    figure_layers.unlink()

prediction_layer.to_file(
    figure_layers,
    layer="predictions",
    driver="GPKG",
)

reference_layer.to_file(
    figure_layers,
    layer="references",
    driver="GPKG",
)

print(
    "Figure 3 layers written:",
    figure_layers,
)

print(
    f"Predictions: {len(prediction_layer)} | "
    f"references: {len(reference_layer)}"
)